# HireSense AI - Resume NER Training (Local GPU)

**Optimized for modern GPUs (e.g., RTX 4060 Ti)**\n

Train a BERT + BiLSTM + CRF model for Named Entity Recognition on resumes.

**Model Architecture:**
- BERT-base-uncased (768-dim contextual embeddings)
- BiLSTM (2 layers, 256 hidden units, bidirectional)
- CRF layer for sequence labeling

**Entity Types:**
- SKILL, EXP, EDU, PROJ, ACH, ORG, LOC, DATE, NAME, CONTACT, CERT

**Output:**
- `./output/best_model.pt` - Best model checkpoint during training
- `./output/resume_ner_model.pt` - Final model for deployment
- `./output/model_config.json` - Model configuration
- `./output/tokenizer/` - Saved tokenizer files

---
**Run each cell in order (Shift+Enter)**

## 1. Setup Environment

In [ ]:
# Install dependencies (run once)
# For RTX 4060Ti with CUDA 11.8+:
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install other dependencies
!pip install -q transformers pytorch-crf seqeval datasets pandas scikit-learn tqdm pdfplumber


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Check GPU and system info
import torch
import platform
import os

print(f"Python version: {platform.python_version()}")
print(f"PyTorch version: {torch.__version__}")
print(f"Platform: {platform.system()} {platform.release()}")
print(f"\nCUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Set memory allocation strategy for better memory management
    torch.cuda.empty_cache()
    print("\nGPU is ready for training!")
else:
    print("\nWARNING: No GPU detected. Training will be slow on CPU.")
    print("For RTX 4060Ti, install PyTorch with CUDA:")
    print("pip install torch --index-url https://download.pytorch.org/whl/cu118")

Python version: 3.11.3
PyTorch version: 2.7.1+cu118
Platform: Windows 10

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
CUDA version: 11.8
VRAM: 17.2 GB

GPU is ready for training!


In [11]:
# Create output directories
OUTPUT_DIR = "./output"
DATA_DIR = "./data"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(f"{DATA_DIR}/kaggle_resume_pdf", exist_ok=True)

print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")
print(f"Data directory: {os.path.abspath(DATA_DIR)}")

Output directory: d:\My Documents\Downloads\HireSense\scripts\training\output
Data directory: d:\My Documents\Downloads\HireSense\scripts\training\data


## 2. Configuration

Optimized for RTX 3060Ti (8GB VRAM):
- Batch size: 8 (fits in 8GB with BERT)
- Gradient accumulation: 4 (effective batch of 32)
- Mixed precision (FP16): Enabled for faster training

In [12]:
# Configuration
from dataclasses import dataclass
import random
import numpy as np

# ============ TRAINING HYPERPARAMETERS ============
# Adjust these based on your GPU VRAM

BATCH_SIZE = 8           # 8 for 8GB VRAM, 16 for 12GB+
ACCUMULATION_STEPS = 4   # Effective batch = BATCH_SIZE * ACCUMULATION_STEPS
EPOCHS = 15              # Number of training epochs
MAX_LENGTH = 512         # Maximum sequence length
LSTM_HIDDEN = 256        # LSTM hidden size
BERT_LR = 2e-5           # BERT learning rate
LSTM_LR = 1e-3           # LSTM/CRF learning rate
USE_FP16 = True          # Mixed precision (set False if issues)
PATIENCE = 5             # Early stopping patience
SEED = 42                # Random seed

# ============ PATHS ============
DATA_PATH = "./data/kaggle_resume_pdf"  # Path to resume data
OUTPUT_PATH = "./output"                 # Output directory

# ============ DEVICE ============
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

print(f"Device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Effective batch size: {BATCH_SIZE * ACCUMULATION_STEPS}")
print(f"Mixed precision (FP16): {USE_FP16}")
print(f"Epochs: {EPOCHS}")

Device: cuda
Batch size: 8
Effective batch size: 32
Mixed precision (FP16): True
Epochs: 15


In [13]:
# Entity labels (BIO format)
ENTITY_LABELS = [
    "O", "B-SKILL", "I-SKILL", "B-EXP", "I-EXP", "B-EDU", "I-EDU",
    "B-PROJ", "I-PROJ", "B-ACH", "I-ACH", "B-ORG", "I-ORG",
    "B-LOC", "I-LOC", "B-DATE", "I-DATE", "B-NAME", "I-NAME",
    "B-CONTACT", "I-CONTACT", "B-CERT", "I-CERT"
]

LABEL2ID = {label: idx for idx, label in enumerate(ENTITY_LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}
NUM_LABELS = len(ENTITY_LABELS)

print(f"Number of entity labels: {NUM_LABELS}")
print(f"Labels: {ENTITY_LABELS}")

Number of entity labels: 23
Labels: ['O', 'B-SKILL', 'I-SKILL', 'B-EXP', 'I-EXP', 'B-EDU', 'I-EDU', 'B-PROJ', 'I-PROJ', 'B-ACH', 'I-ACH', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-DATE', 'I-DATE', 'B-NAME', 'I-NAME', 'B-CONTACT', 'I-CONTACT', 'B-CERT', 'I-CERT']


## 3. Model Architecture

**BERT + BiLSTM + CRF**
- BERT: Contextual embeddings (768-dim)
- BiLSTM: Sequence modeling (2 layers, 256 hidden)
- CRF: Sequence labeling with transition constraints

In [14]:
import torch.nn as nn
from transformers import BertModel
from torchcrf import CRF

class BertBiLSTMCRF(nn.Module):
    """BERT + BiLSTM + CRF for Resume NER"""
    
    def __init__(self, bert_model_name="bert-base-uncased", lstm_hidden=256, lstm_layers=2, dropout=0.3):
        super().__init__()
        
        # BERT encoder
        self.bert = BertModel.from_pretrained(bert_model_name)
        bert_hidden = self.bert.config.hidden_size
        
        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=bert_hidden,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0
        )
        
        self.dropout = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(lstm_hidden * 2, NUM_LABELS)
        self.crf = CRF(NUM_LABELS, batch_first=True)
        
    def forward(self, input_ids, attention_mask, labels=None):
        # BERT encoding
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = bert_output.last_hidden_state
        
        # BiLSTM
        lstm_output, _ = self.lstm(sequence_output)
        lstm_output = self.dropout(lstm_output)
        
        # Emission scores
        emissions = self.hidden2tag(lstm_output)
        
        outputs = {"emissions": emissions}
        
        # Calculate loss if labels provided
        if labels is not None:
            mask = (labels != -100) & (attention_mask == 1)
            labels_crf = labels.clone()
            labels_crf[labels == -100] = 0
            loss = -self.crf(emissions, labels_crf, mask=mask, reduction='mean')
            outputs["loss"] = loss
        
        # CRF decoding
        with torch.no_grad():
            predictions = self.crf.decode(emissions, mask=attention_mask.bool())
            outputs["predictions"] = predictions
            
        return outputs

# Initialize model
print("Loading BERT model (this may take a minute)...")
model = BertBiLSTMCRF(lstm_hidden=LSTM_HIDDEN)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model loaded on: {DEVICE}")

Loading BERT model (this may take a minute)...

Total parameters: 113,172,822
Trainable parameters: 113,172,822
Model loaded on: cuda


## 4. Dataset Preparation

**Option A:** Place your resume PDFs/TXTs in `./data/kaggle_resume_pdf/`

**Option B:** Uses synthetic data if no files found (for testing)

In [15]:
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast
from sklearn.model_selection import train_test_split
import glob
import re

# Load tokenizer
print("Loading tokenizer...")
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

class ResumeNERDataset(Dataset):
    """PyTorch Dataset for Resume NER"""
    
    def __init__(self, examples, tokenizer, max_length=512):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        tokens, labels = self.examples[idx]
        
        encoding = self.tokenizer(
            tokens, is_split_into_words=True,
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt"
        )
        
        # Align labels with tokenized input
        word_ids = encoding.word_ids()
        aligned_labels = []
        prev_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)  # Special tokens
            elif word_idx != prev_word_idx:
                if word_idx < len(labels):
                    aligned_labels.append(LABEL2ID.get(labels[word_idx], 0))
                else:
                    aligned_labels.append(0)
            else:
                # Subword: use I- tag if B- tag
                if word_idx < len(labels):
                    label = labels[word_idx]
                    if label.startswith("B-"):
                        label = "I-" + label[2:]
                    aligned_labels.append(LABEL2ID.get(label, 0))
                else:
                    aligned_labels.append(0)
            prev_word_idx = word_idx
        
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(aligned_labels, dtype=torch.long)
        }

print("Dataset class ready.")

Loading tokenizer...
Dataset class ready.


In [ ]:
# Load resume data from files OR generate synthetic data
from tqdm import tqdm

import os
import re
import random
from tqdm import tqdm

def load_resume_files(data_path):
    """Load resumes from PDF/TXT files with strong OCR fallback (FIXED)"""
    
    examples = []
    stats = {
        "pdf_found": 0,
        "txt_found": 0,
        "pdf_success": 0,
        "txt_success": 0,
        "ocr_used": 0,
        "empty": 0,
        "errors": 0
    }

    # 🔥 Skill vocabulary
    skill_terms = {
        "python", "javascript", "java", "c++", "react", "node",
        "pytorch", "tensorflow", "sql", "mongodb", "docker", "aws"
    }

    def weak_label_text(text):
        """LESS STRICT labeling (FIXED)"""
        
        if not text or len(text.strip()) < 5:
            return None

        tokens = re.findall(r"\b\w+[\w+.+-]*\b|[^\w\s]", text)

        if len(tokens) < 3:
            return None

        labels = ["O"] * len(tokens)

        email_pattern = re.compile(r"[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}")

        for i, tok in enumerate(tokens):
            tok_lower = tok.lower()

            if tok_lower in skill_terms:
                labels[i] = "B-SKILL"

            elif email_pattern.fullmatch(tok):
                labels[i] = "B-CONTACT"

        return (tokens, labels)

    # ===== scan files =====
    pdf_files, txt_files = [], []

    for root, _, files in os.walk(data_path):
        for file in files:
            path = os.path.join(root, file)
            if file.lower().endswith(".pdf"):
                pdf_files.append(path)
            elif file.lower().endswith(".txt"):
                txt_files.append(path)

    stats["pdf_found"] = len(pdf_files)
    stats["txt_found"] = len(txt_files)

    print(f"Found {len(pdf_files)} PDF files and {len(txt_files)} TXT files")

    # ===== TXT =====
    for filepath in tqdm(txt_files, desc="Loading TXT files"):
        try:
            with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()

            result = weak_label_text(text)
            if result:
                examples.append(result)
                stats["txt_success"] += 1
        except:
            stats["errors"] += 1

    # ===== PDF =====
    import pdfplumber

    try:
        from pdf2image import convert_from_path
        import pytesseract

        pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
        HAS_OCR = True
        print("OCR ENABLED ✅")
    except:
        HAS_OCR = False
        print("OCR NOT AVAILABLE ❌")

    for filepath in tqdm(pdf_files, desc="Loading PDF files"):
        try:
            text = ""

            # 🔥 TRY pdfplumber
            try:
                with pdfplumber.open(filepath) as pdf:
                    for page in pdf.pages:
                        t = page.extract_text()
                        if t:
                            text += t + "\n"
            except:
                pass

            # 🔥 FORCE OCR if weak text (MAIN FIX)
            if len(text.strip()) < 30 and HAS_OCR:
                try:
                    images = convert_from_path(filepath, dpi=200, first_page=1, last_page=2)
                    ocr_text = ""

                    for img in images:
                        ocr_text += pytesseract.image_to_string(img) + "\n"

                    if ocr_text.strip():
                        text = ocr_text
                        stats["ocr_used"] += 1
                except:
                    pass

            if not text.strip():
                stats["empty"] += 1
                continue

            result = weak_label_text(text)

            if result is None:
                continue

            examples.append(result)
            stats["pdf_success"] += 1

        except:
            stats["errors"] += 1

    # ===== STATS =====
    print("\n--- Loading Statistics ---")
    for k, v in stats.items():
        print(f"{k}: {v}")
    print(f"Total usable resumes loaded: {len(examples)}")
    print("--------------------------\n")

    return examples


def generate_synthetic_data(num_examples=2000):
    """Generate synthetic training data"""
    skills = ["Python", "JavaScript", "React", "Node.js", "TypeScript", "Java",
              "C++", "SQL", "MongoDB", "PostgreSQL", "AWS", "Docker", "Kubernetes",
              "TensorFlow", "PyTorch", "Machine Learning", "Deep Learning", "NLP"]
    companies = ["Google", "Microsoft", "Amazon", "Meta", "Apple", "Netflix"]
    titles = ["Software Engineer", "Senior Developer", "Data Scientist", "ML Engineer"]
    universities = ["MIT", "Stanford", "UC Berkeley", "Carnegie Mellon"]
    degrees = ["Bachelor of Science in Computer Science", "Master of Science in Data Science"]
    
    examples = []
    for _ in range(num_examples):
        tokens, labels = [], []
        
        # Name
        tokens.extend(["John", "Doe"])
        labels.extend(["B-NAME", "I-NAME"])
        tokens.append("|")
        labels.append("O")
        
        # Skills
        tokens.extend(["Skills", ":"])
        labels.extend(["O", "O"])
        for i, skill in enumerate(random.sample(skills, random.randint(3, 6))):
            for j, s in enumerate(skill.split()):
                tokens.append(s)
                labels.append("B-SKILL" if j == 0 else "I-SKILL")
            if i < 5:
                tokens.append(",")
                labels.append("O")
        
        # Experience
        tokens.extend(["Experience", ":"])
        labels.extend(["O", "O"])
        title = random.choice(titles)
        for j, t in enumerate(title.split()):
            tokens.append(t)
            labels.append("B-EXP" if j == 0 else "I-EXP")
        tokens.append("at")
        labels.append("O")
        tokens.append(random.choice(companies))
        labels.append("B-ORG")
        
        # Education
        tokens.extend(["Education", ":"])
        labels.extend(["O", "O"])
        degree = random.choice(degrees)
        for j, d in enumerate(degree.split()):
            tokens.append(d)
            labels.append("B-EDU" if j == 0 else "I-EDU")
        
        examples.append((tokens, labels))
    
    return examples


# Load data
print(f"Looking for data in: {os.path.abspath(DATA_PATH)}")
all_examples = load_resume_files(DATA_PATH)
print(f"Loaded {len(all_examples)} examples from files")

# Add synthetic data if needed
if len(all_examples) < 500:
    print(f"Adding synthetic data to reach minimum training size...")
    synthetic = generate_synthetic_data(2000 - len(all_examples))
    all_examples.extend(synthetic)

print(f"Total examples: {len(all_examples)}")

Looking for data in: d:\My Documents\Downloads\HireSense\scripts\training\data\kaggle_resume_pdf
Found 8833 PDF files and 0 TXT files


Loading TXT files: 0it [00:00, ?it/s]


OCR support available (pdf2image + pytesseract)


Loading PDF files: 100%|██████████| 8833/8833 [01:20<00:00, 109.77it/s]


--- Loading Statistics ---
PDF files found: 8833
PDF successfully loaded: 0
PDF via OCR: 0
TXT files found: 0
TXT successfully loaded: 0
Empty (no text): 8773
Errors: 60
--------------------------

Loaded 0 examples from files
Adding synthetic data to reach minimum training size...
Total examples: 2000


In [ ]:
# Split data into train/val/test
train_examples, temp = train_test_split(all_examples, test_size=0.2, random_state=SEED)
val_examples, test_examples = train_test_split(temp, test_size=0.5, random_state=SEED)

print(f"Train: {len(train_examples)}")
print(f"Val: {len(val_examples)}")
print(f"Test: {len(test_examples)}")

# Create datasets
train_dataset = ResumeNERDataset(train_examples, tokenizer, MAX_LENGTH)
val_dataset = ResumeNERDataset(val_examples, tokenizer, MAX_LENGTH)
test_dataset = ResumeNERDataset(test_examples, tokenizer, MAX_LENGTH)

# DataLoaders (num_workers=0 for Windows compatibility)
num_workers = 0 if platform.system() == "Windows" else 4

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=num_workers, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE * 2,
    num_workers=num_workers, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE * 2,
    num_workers=num_workers, pin_memory=True
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 5. Training Setup

- Different learning rates for BERT vs LSTM/CRF
- Mixed precision (FP16) for faster training
- Gradient accumulation for larger effective batch size

In [ ]:
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from seqeval.metrics import f1_score, classification_report

# Separate BERT and LSTM/CRF parameters (different learning rates)
bert_params = [p for n, p in model.named_parameters() if 'bert' in n and p.requires_grad]
other_params = [p for n, p in model.named_parameters() if 'bert' not in n and p.requires_grad]

optimizer = AdamW([
    {'params': bert_params, 'lr': BERT_LR},
    {'params': other_params, 'lr': LSTM_LR}
], weight_decay=0.01)

# Learning rate scheduler with warmup
num_training_steps = len(train_loader) * EPOCHS // ACCUMULATION_STEPS
num_warmup_steps = int(num_training_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=num_warmup_steps, 
    num_training_steps=num_training_steps
)

# Mixed precision scaler
scaler = GradScaler() if USE_FP16 and torch.cuda.is_available() else None

print(f"Optimizer: AdamW")
print(f"BERT LR: {BERT_LR}")
print(f"LSTM/CRF LR: {LSTM_LR}")
print(f"Total training steps: {num_training_steps}")
print(f"Warmup steps: {num_warmup_steps}")
print(f"Mixed precision: {scaler is not None}")

In [ ]:
def evaluate(model, dataloader, device):
    """Evaluate model and return metrics"""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(input_ids, attention_mask, labels)
            total_loss += outputs["loss"].item()
            
            # Collect predictions
            for pred_seq, label_seq, mask in zip(
                outputs["predictions"], labels.cpu().numpy(), attention_mask.cpu().numpy()
            ):
                pred_labels, true_labels = [], []
                for pred, label, m in zip(pred_seq, label_seq, mask):
                    if m == 1 and label != -100:
                        pred_labels.append(ID2LABEL[pred])
                        true_labels.append(ID2LABEL[label])
                if pred_labels:
                    all_preds.append(pred_labels)
                    all_labels.append(true_labels)
    
    return {
        "loss": total_loss / len(dataloader) if dataloader else 0,
        "f1": f1_score(all_labels, all_preds) if all_labels else 0,
        "preds": all_preds,
        "labels": all_labels
    }

print("Evaluation function ready.")

## 6. Training Loop

**This will take ~20-40 minutes on an RTX 4060 Ti**\n

Progress will show:
- Loss per batch
- GPU memory usage
- Validation F1 after each epoch

In [ ]:
# Training loop
print("="*60)
print("STARTING TRAINING")
print("="*60)
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * ACCUMULATION_STEPS})")
print(f"Mixed precision: {USE_FP16}")
print("="*60)

best_f1 = 0
patience_counter = 0
history = []
use_amp = USE_FP16 and torch.cuda.is_available()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    
    for batch_idx, batch in enumerate(progress):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        
        # Forward pass with mixed precision
        if use_amp:
            with autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(input_ids, attention_mask, labels)
                loss = outputs["loss"] / ACCUMULATION_STEPS
            
            scaler.scale(loss).backward()
            
            if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
        else:
            outputs = model(input_ids, attention_mask, labels)
            loss = outputs["loss"] / ACCUMULATION_STEPS
            loss.backward()
            
            if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
        
        total_loss += loss.item() * ACCUMULATION_STEPS
        
        # Update progress bar
        if torch.cuda.is_available():
            gpu_mem = torch.cuda.memory_allocated() / 1e9
            progress.set_postfix({
                "loss": f"{loss.item() * ACCUMULATION_STEPS:.4f}",
                "gpu": f"{gpu_mem:.1f}GB"
            })
        else:
            progress.set_postfix({"loss": f"{loss.item() * ACCUMULATION_STEPS:.4f}"})
    
    # Evaluate on validation set
    val_results = evaluate(model, val_loader, DEVICE)
    avg_train_loss = total_loss / len(train_loader)
    
    print(f"\nEpoch {epoch}: Train Loss={avg_train_loss:.4f}, Val Loss={val_results['loss']:.4f}, Val F1={val_results['f1']:.4f}")
    
    history.append({
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": val_results["loss"],
        "val_f1": val_results["f1"]
    })
    
    # Save best model
    if val_results["f1"] > best_f1:
        best_f1 = val_results["f1"]
        patience_counter = 0
        
        # Save checkpoint
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "label2id": LABEL2ID,
            "id2label": ID2LABEL,
            "f1": best_f1
        }, os.path.join(OUTPUT_PATH, "best_model.pt"))
        
        print(f"  -> Saved best model to {OUTPUT_PATH}/best_model.pt (F1: {best_f1:.4f})")
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

print("\n" + "="*60)
print(f"Training complete! Best Val F1: {best_f1:.4f}")
print("="*60)

## 7. Final Evaluation on Test Set

In [ ]:
# Load best model
print("Loading best model...")
checkpoint = torch.load(os.path.join(OUTPUT_PATH, "best_model.pt"))
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded model from epoch {checkpoint['epoch']} with F1: {checkpoint['f1']:.4f}")

# Evaluate on test set
print("\nEvaluating on test set...")
test_results = evaluate(model, test_loader, DEVICE)

print("\n" + "="*60)
print("TEST RESULTS")
print("="*60)
print(f"Test F1: {test_results['f1']:.4f}")
print(f"Test Loss: {test_results['loss']:.4f}")
print("\nClassification Report:")
print(classification_report(test_results["labels"], test_results["preds"]))

## 8. Test Entity Extraction

In [ ]:
def extract_entities(model, tokenizer, text, device=DEVICE):
    """Extract entities from raw text"""
    model.eval()
    tokens = text.split()
    
    encoding = tokenizer(
        tokens, is_split_into_words=True,
        max_length=512, padding="max_length", truncation=True, return_tensors="pt"
    )
    
    with torch.no_grad():
        input_ids = encoding["input_ids"].to(device)
        attention_mask = encoding["attention_mask"].to(device)
        outputs = model(input_ids, attention_mask)
    
    # Extract entities from predictions
    predictions = outputs["predictions"][0]
    word_ids = encoding.word_ids()
    
    entities = []
    current_entity = None
    prev_word_idx = None
    
    for idx, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx == prev_word_idx:
            prev_word_idx = word_idx
            continue
        
        if idx >= len(predictions):
            break
            
        label = ID2LABEL.get(predictions[idx], "O")
        
        if label.startswith("B-"):
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                "text": tokens[word_idx] if word_idx < len(tokens) else "",
                "label": label[2:],
                "start": word_idx
            }
        elif label.startswith("I-") and current_entity and label[2:] == current_entity["label"]:
            if word_idx < len(tokens):
                current_entity["text"] += " " + tokens[word_idx]
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None
        
        prev_word_idx = word_idx
    
    if current_entity:
        entities.append(current_entity)
    
    return entities

# Test extraction
sample_resume = """
John Smith Software Engineer at Google with 5 years of experience
Skills Python JavaScript React TensorFlow AWS Docker Kubernetes
Education Master of Science in Computer Science from Stanford University 2018
Certifications AWS Certified Solutions Architect
"""

print("Sample Resume:")
print(sample_resume)
print("\nExtracted Entities:")
entities = extract_entities(model, tokenizer, sample_resume)
for e in entities:
    print(f"  {e['label']}: {e['text']}")

## 9. Export Model for Deployment

Saves:
- `resume_ner_model.pt` - Model weights
- `model_config.json` - Model configuration
- `tokenizer/` - Tokenizer files

In [ ]:
import json

# Save final model for deployment
print("Exporting model for deployment...")

# Save model weights
model_path = os.path.join(OUTPUT_PATH, "resume_ner_model.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "label2id": LABEL2ID,
    "id2label": ID2LABEL,
    "num_labels": NUM_LABELS,
    "f1": test_results["f1"]
}, model_path)
print(f"Model saved to: {model_path}")

# Save model config
config_path = os.path.join(OUTPUT_PATH, "model_config.json")
with open(config_path, "w") as f:
    json.dump({
        "bert_model_name": "bert-base-uncased",
        "lstm_hidden_size": LSTM_HIDDEN,
        "lstm_num_layers": 2,
        "lstm_dropout": 0.3,
        "num_labels": NUM_LABELS,
        "max_seq_length": MAX_LENGTH,
        "label2id": LABEL2ID,
        "id2label": {str(k): v for k, v in ID2LABEL.items()}
    }, f, indent=2)
print(f"Config saved to: {config_path}")

# Save tokenizer
tokenizer_path = os.path.join(OUTPUT_PATH, "tokenizer")
tokenizer.save_pretrained(tokenizer_path)
print(f"Tokenizer saved to: {tokenizer_path}")

# Print file sizes
model_size = os.path.getsize(model_path) / 1e6
print(f"\nModel size: {model_size:.1f} MB")

print("\n" + "="*60)
print("EXPORT COMPLETE!")
print("="*60)
print(f"\nFiles saved to: {os.path.abspath(OUTPUT_PATH)}")
print("\nFiles:")
for f in os.listdir(OUTPUT_PATH):
    fpath = os.path.join(OUTPUT_PATH, f)
    if os.path.isfile(fpath):
        print(f"  - {f} ({os.path.getsize(fpath) / 1e6:.1f} MB)")
    else:
        print(f"  - {f}/ (directory)")

## 10. Training History Visualization

In [ ]:
import matplotlib.pyplot as plt

if history:
    epochs = [h["epoch"] for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss = [h["val_loss"] for h in history]
    val_f1 = [h["val_f1"] for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, train_loss, label="Train", marker='o')
    ax1.plot(epochs, val_loss, label="Val", marker='o')
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.set_title("Training & Validation Loss")
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, val_f1, color='green', marker='o')
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("F1 Score")
    ax2.set_title("Validation F1 Score")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    
    # Save figure
    fig_path = os.path.join(OUTPUT_PATH, "training_history.png")
    plt.savefig(fig_path, dpi=150)
    print(f"Training history saved to: {fig_path}")
    
    plt.show()
else:
    print("No training history to plot.")

## Done!

Your trained model is saved in the `./output/` directory:

- **resume_ner_model.pt** - Use this for deployment
- **model_config.json** - Model architecture settings
- **tokenizer/** - Required for inference

To use in the HireSense AI app:
1. Copy these files to the `backend/models/` directory
2. The FastAPI backend will automatically load the model